In [2]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()


D:\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "description": "Over-ear Bluestooth, 30-hrs battery, active noise cancellation." },
    "smart watch":         {"price": 199.99, "description": "tracks heart rate and sleep. 5-day battery, water-resistant"},
    "mechanical keyboard": {"price": 129.00, "description": "Tenkeyless, cherry MX brown switches, per-key RGB"},
    "laptop stand":        {"price": 34.99,  "description": "Adjustable alluminium, fits 11-17 inch laptops, folds flat"}
}

@tool
def get_product(name: str) ->str:
    """ Look up a product by name and return its price, rating, stock and descritpion. """ 
    p= PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available produts are : {'.'.join(PRODUCTS)}"
    return str(p)
    
# llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature'0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="you are a helpful product assistant for an online tech store."
)

In [4]:
def ask(question:str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99. They are over-ear Bluetooth headphones with 30 hours of battery life and active noise cancellation.


In [6]:
ask("what are the reviews of this product")

Based on the product information, here are the reviews for each product:

1. Wireless Headphones: 4.5/5 stars, "Great sound quality and comfortable to wear. Battery life is impressive."
2. Smart Watch: 4.2/5 stars, "Easy to use and tracks my fitness goals accurately. Battery life could be better."
3. Mechanical Keyboard: 4.8/5 stars, "Love the tactile feedback and customizable RGB lighting. Worth the investment."
4. Laptop Stand: 4.5/5 stars, "Sturdy and adjustable, perfect for working on the go. Folds up nicely for storage."

Please note that these reviews are fictional and for demonstration purposes only.


In [ ]:
# 1. extract product name from question ("what is the price of wireless headphones.")
# 2. calls get_product(" wireless headphone")
# 3. analyze the answer {"price": 79.99,  "description": "Over-ear Bluestooth, 30-hrs battery, active noise cancellation." }
# 4. return human readable answer.

In [7]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating":4.6},
    "smart watch":        {"reviews": 340, "rating":3.9},
    "mechanical keyboard":{"reviews": 67, "rating":4.8},
    "laptop stand":       {"reviews": 781, "rating":4.5}
}

@tool
def get_review(name: str) -> str:
    """ Look up a product review by product name. Return the product name, number of reviews and rating """
    r = REVIEWS.get(name.lower())
    if not r:
        print(f" Review not available for the given product.")
    return str(r)

In [8]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="you are a helpful product assistant for an online tech store."
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [9]:
ask2("how do people like smart watch")

People like smart watches because they are convenient and provide a range of useful features such as tracking heart rate and sleep, and being water-resistant. The smart watch has a rating of 3.9 out of 5 stars based on 340 reviews, and is priced at $199.99.


In [10]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
agent3 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="you are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask3(question: str):
    config = {"configurable":{"thread_id":"user-alice-session-1"}}
    result = agent3.invoke({"messages": [{"role": "user", "content": question}]}, config)
    print(result["messages"][-1].content)

In [11]:
ask3("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [12]:
ask3("what are the reviews on this product?")

The wireless headphones have 1262 reviews with an average rating of 4.6.
